In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)

## Data

In [ ]:
def sample_x0(n, D):
    return torch.randn(n, D)

In [ ]:
def sample_x1(n, weights, means, stds):
    k = torch.multinomial(weights, n, replacement=True)
    return means[k] + stds[k, None] * torch.randn(n, means.shape[1])

1D Mixture gaussian

In [ ]:
weights = torch.tensor([0.5, 0.5])
means = torch.tensor([[1.0], [-1.0]])
stds = torch.tensor([0.3, 0.3])

n = 500
x_1 = sample_x1(n, weights, means, stds)

$$
p(x) = \sum_k w_k \,\mathcal N(x;\,\mu_k, \sigma_k^2),\qquad \mathcal N(x;\,\mu, \sigma^2) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac12\Big(\frac{x-\mu}{\sigma}\Big)^2\right)
$$

In [ ]:
x = torch.linspace(-4, 4, 1000)
mu = means[:, 0] # (2, 1) -> (2,)
dens = (weights * torch.exp(-0.5 * ((x[:, None] - mu) / stds) ** 2)
        / (stds * (2 * torch.pi) ** 0.5)).sum(dim=1)

plt.hist(x_1[:, 0], bins=60, density=True, alpha=0.5, label="samples")
plt.plot(x, dens, label="true density")
plt.legend()
plt.show()

## 理想經驗場

訓練時隨機抽一個起點和一個終點連成直線，讓網路在線上某處預測這條線的方向。

$$
\mathcal{L}_{\text{RF}}(\theta) = \mathbb{E}_{\,t \sim \text{Unif}[0,1],\; x_0 \sim \mathcal N(0, I),\; x_1 \sim p_{\text{data}}} \Big[\, \big\| v_\theta(x_t, t) - (x_1 - x_0) \big\|^2 \,\Big]
$$

同一個位置會有很多條不同的線經過，各自指向不同方向，網路只能給一個答案。平方誤差最小的答案就是平均，所以網路最終學到的是經過這裡的所有線的平均方向。

- 對每個終點 $x_1$ 與給定的當下位置 $x_t$，反推出對應的起點 $x_0$ 與速度 $x_1 - x_0$
- 用起點的機率當權重，再平均各配對的速度

0. 設定：起點 $x_0 \sim \mathcal N(0, I)$，終點 $x_1$ 從訓練集 $\{x_1^{(1)},\dots,x_1^{(N)}\}$ 中均勻抽取

1. 給定當下位置 $x_t$、時間 $t$，對於每個終點 $x^{(i)}_1$，反推起點 $x_0$

$$
\begin{aligned}
x_t &= t\,x^{(i)}_1 + (1-t)\,x_0 \\
\implies x_0 &= \frac{x_t - t\,x^{(i)}_1}{1 - t} 
\end{aligned}
$$

2. 每一對的速度

$$
\begin{aligned}
x^{(i)}_1 - x_0 = x^{(i)}_1 - \frac{x_t - t\,x^{(i)}_1}{1 - t} = \frac{x^{(i)}_1 - x_t}{1 - t} 
\end{aligned}
$$

3. 每一對速度的起點在初始分布 $\mathcal N(0,I)$ 中的機率

In [ ]:
def empirical_v(x_t, t, data):
    # x_t: (B, D), data: (N, D)
    sq = torch.cdist(x_t, t * data) ** 2      # (B, N)
    w = torch.softmax(-sq / (2 * (1 - t) ** 2), dim=1)
    v_t = (w @ data - x_t) / (1 - t)
    return v_t

In [ ]:
def gmm_v(x_t, t, weights, means, stds):
    # x_t: (B, D), means: (K, D), weights/stds: (K,)
    D = x_t.shape[1]
    a = (1 - t) ** 2 + t ** 2 * stds ** 2                                  # (K,)
    sq = ((x_t[:, None, :] - t * means[None]) ** 2).sum(dim=2)             # (B, K)
    r = torch.softmax(torch.log(weights) - 0.5 * D * torch.log(a) - sq / (2 * a), dim=1)
    cond = means[None] + (t * stds ** 2 / a)[None, :, None] * (x_t[:, None, :] - t * means[None])
    return ((r[:, :, None] * cond).sum(dim=1) - x_t) / (1 - t)            # (B, D)

In [ ]:
xs = torch.linspace(-4, 4, 1000)[:, None]          # 查詢點 (B, 1)

ts = [0.2, 0.4, 0.6, 0.8, 0.95]
fig, axes = plt.subplots(1, len(ts), figsize=(16, 3.5))
for ax, t in zip(axes, ts):
    ax.plot(xs[:, 0], gmm_v(xs, t, weights, means, stds)[:, 0], label="gmm")
    ax.plot(xs[:, 0], empirical_v(xs, t, x_1)[:, 0], label="empirical", alpha=0.8)
    ax.set(title=f"t = {t}", xlabel="x_t", ylim=(-10, 10))
axes[0].legend()
plt.show()

In [ ]:
def gmm_pt(x_t, t, weights, means, stds):
    D = x_t.shape[1]
    a = (1 - t) ** 2 + t ** 2 * stds ** 2
    sq = ((x_t[:, None, :] - t * means[None]) ** 2).sum(dim=2)
    return (weights * torch.exp(-sq / (2 * a)) / (2 * torch.pi * a) ** (D / 2)).sum(dim=1)

ts = torch.linspace(0.01, 0.99, 99)
err = []
for t in ts.tolist():
    ve = empirical_v(xs, t, x_1)
    vg = gmm_v(xs, t, weights, means, stds)
    p = gmm_pt(xs, t, weights, means, stds)
    p = p / p.sum()
    num = (p * ((ve - vg) ** 2).sum(dim=1)).sum()
    den = (p * (vg ** 2).sum(dim=1)).sum()
    err.append((num / den).sqrt().item())

plt.plot(ts, err)
plt.xlabel("t"); plt.ylabel("relative error")
plt.show()

In [ ]:
def fit_moments(x1):
    mu = x1.mean(dim=0)                       # (D,)
    r = x1 - mu
    Sigma = r.T @ r / len(x1)                 # (D, D)
    return mu, Sigma

def moment_v(x_t, t, mu, Sigma):
    D = x_t.shape[1]
    A = (1 - t) ** 2 * torch.eye(D) + t ** 2 * Sigma
    m = mu + t * (x_t - t * mu) @ torch.linalg.solve(A, Sigma).T
    return (m - x_t) / (1 - t)

In [ ]:
mu, Sigma = fit_moments(x_1)

In [ ]:
ts = [0.2, 0.4, 0.6, 0.8, 0.95]
fig, axes = plt.subplots(1, len(ts), figsize=(16, 3.5))
for ax, t in zip(axes, ts):
    ax.plot(xs[:, 0], gmm_v(xs, t, weights, means, stds)[:, 0], label="population")
    ax.plot(xs[:, 0], empirical_v(xs, t, x_1)[:, 0], label="empirical", alpha=0.8)
    ax.plot(xs[:, 0], moment_v(xs, t, mu, Sigma)[:, 0], label="moment", ls="--")
    ax.set(title=f"t = {t}", xlabel="x_t", ylim=(-10, 10))
axes[0].legend()
plt.show()

In [ ]:
def rel_err(va, vb, p):
    num = (p * ((va - vb) ** 2).sum(dim=1)).sum()
    den = (p * (vb ** 2).sum(dim=1)).sum()
    return (num / den).sqrt().item()

ts = torch.linspace(0.01, 0.99, 99)
errs = {"empirical vs population": [], "moment vs population": [], "moment vs empirical": []}
for t in ts.tolist():
    ve = empirical_v(xs, t, x_1)
    vp = gmm_v(xs, t, weights, means, stds)
    vm = moment_v(xs, t, mu, Sigma)
    p = gmm_pt(xs, t, weights, means, stds)
    p = p / p.sum()
    errs["empirical vs population"].append(rel_err(ve, vp, p))
    errs["moment vs population"].append(rel_err(vm, vp, p))
    errs["moment vs empirical"].append(rel_err(vm, ve, p))

for name, e in errs.items():
    plt.plot(ts, e, label=name)
plt.xlabel("t"); plt.ylabel("relative error"); plt.legend()
plt.show()

In [ ]:
def integrate(v_fn, x0, steps=200):
    x, traj = x0.clone(), [x0.clone()]
    for i in range(steps):
        x = x + v_fn(x, i / steps) / steps
        traj.append(x.clone())
    return torch.stack(traj)                     # (steps + 1, B, D)

fields = {
    "Population field": lambda x, t: gmm_v(x, t, weights, means, stds),
    f"Empirical field (N = {len(x_1)})": lambda x, t: empirical_v(x, t, x_1),
    "Moment field (single Gaussian)": lambda x, t: moment_v(x, t, mu, Sigma),
}

x0_lines = sample_x0(60, 1).sort(dim=0).values   # 畫軌跡用
x0_hist = sample_x0(4000, 1)                     # 算終點分布用
ts = torch.linspace(0, 1, 201)

grid = torch.linspace(-3.5, 3.5, 600)
src = torch.exp(-grid ** 2 / 2) / (2 * torch.pi) ** 0.5
tgt = (weights * torch.exp(-0.5 * ((grid[:, None] - means[:, 0]) / stds) ** 2)
       / (stds * (2 * torch.pi) ** 0.5)).sum(dim=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, (name, v_fn) in zip(axes, fields.items()):
    traj = integrate(v_fn, x0_lines)[:, :, 0]
    ends = integrate(v_fn, x0_hist)[-1, :, 0]

    ax.plot(ts, traj, color="#2563eb", lw=0.8, alpha=0.45)

    s = 0.18 / src.max()                         # 左：初始分布
    ax.fill_betweenx(grid, 0, -src * s, color="#dbeafe", ec="#60a5fa")

    hist, edges = torch.histogram(ends, bins=70, range=(-3.5, 3.5), density=True)
    centers = (edges[1:] + edges[:-1]) / 2
    h = 0.18 / tgt.max()                         # 右：生成分布與真實密度
    ax.fill_betweenx(centers, 1, 1 + hist * h, step="mid", color="#fed7aa", ec="#ea580c")
    ax.plot(1 + tgt * h, grid, color="#6b7280", ls="--", lw=1.2)

    ax.axvline(0, color="#9ca3af", lw=1)
    ax.axvline(1, color="#9ca3af", lw=1)
    ax.set(title=name, xlim=(-0.22, 1.24), ylim=(-3.2, 3.2),
           xticks=[0, 1], xticklabels=["t = 0", "t = 1"])

axes[0].set_ylabel("x")
plt.tight_layout()
plt.show()